<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [1]:
import pandas as pd
import psycopg2
import requests
import io

In [2]:
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = 'cCkxxLVrDE8EbvjueeMedPKt'
HOST = '84.201.134.129'
PORT = 5432

In [3]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

In [4]:
import warnings

warnings.filterwarnings('ignore')

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [5]:
query_3_1 = '''
    SELECT 
        COUNT(*) 
    FROM 
        vacancies
'''

In [6]:
df = pd.read_sql_query(query_3_1, connection)
df

,count
0,49197


2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [7]:
query_3_2 = '''
    SELECT 
        COUNT(*) 
    FROM 
        employers
'''

In [8]:
df = pd.read_sql_query(query_3_2, connection)
df

,count
0,23501


3. Посчитате с помощью запроса количество регионов (таблица areas).

In [9]:
query_3_3 = '''
    SELECT 
        COUNT(*) 
    FROM 
        areas
'''

In [10]:
df = pd.read_sql_query(query_3_3, connection)
df

,count
0,1362


4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [11]:
query_3_4 = '''
    SELECT 
        COUNT(*) 
    FROM 
        industries
'''

In [12]:
df = pd.read_sql_query(query_3_4, connection)
df

,count
0,294


***

Масштаб: В базе содержится около 49 тысяч вакансий, что является достаточной выборкой для построения модели машинного обучения.
Работодатели: Представлено более 23 тысяч компаний, что говорит о высокой конкуренции на рынке и разнообразии предложений.
География: Вакансии распределены по 1362 регионам, что подтверждает широкий географический охват (включая страны СНГ и зарубежье).
Сферы деятельности: Наличие 294 индустрий позволяет проводить глубокую сегментацию вакансий по направлениям бизнеса.

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [13]:
query_4_1 = '''
    SELECT 
        a.name AS area, 
        COUNT(v.id) AS cnt
    FROM 
        vacancies v
    JOIN 
        areas a ON v.area_id = a.id
    GROUP BY 
        a.name
    ORDER BY 
        cnt DESC
'''

In [14]:
df = pd.read_sql_query(query_4_1, connection)
df

,area,cnt
0,Москва,5333
1,Санкт-Петербург,2851
2,Минск,2112
3,Новосибирск,2006
4,Алматы,1892
...,...,...
764,Тарко-Сале,1
765,Новоаннинский,1
766,Бирск,1
767,Сасово,1


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [15]:
query_4_2 = '''
    SELECT 
        COUNT(*) 
    FROM 
        vacancies
    WHERE 
        salary_from IS NOT NULL 
        OR 
        salary_to IS NOT NULL
'''

In [16]:
df = pd.read_sql_query(query_4_2, connection)
df

,count
0,24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [17]:
query_4_3 = '''
    SELECT 
        ROUND(AVG(salary_from)) AS avg_salary_from, 
        ROUND(AVG(salary_to)) AS avg_salary_to
    FROM 
        vacancies
'''

In [18]:
df = pd.read_sql_query(query_4_3, connection)
df

,avg_salary_from,avg_salary_to
0,71065.0,110537.0


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [19]:
query_4_4 = '''
    SELECT 
        schedule, 
        employment, 
        COUNT(*) AS cnt
    FROM 
        vacancies
    GROUP BY 
        schedule, 
        employment
    ORDER BY 
        cnt DESC
'''

In [20]:
df = pd.read_sql_query(query_4_4, connection)
df

,schedule,employment,cnt
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940
5,Полный день,Стажировка,569
6,Вахтовый метод,Полная занятость,367
7,Полный день,Частичная занятость,347
8,Гибкий график,Частичная занятость,312
9,Полный день,Проектная работа,141


5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [21]:
query_4_5 = '''
    SELECT 
        experience, 
        COUNT(*) AS cnt
    FROM 
        vacancies
    GROUP BY 
        experience
    ORDER BY 
        cnt ASC
'''

In [22]:
df = pd.read_sql_query(query_4_5, connection)
df

,experience,cnt
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

География: Лидером по количеству вакансий являются крупнейшие мегаполисы (Москва, Санкт-Петербург, Минск), что подтверждает концентрацию IT-рынка в столицах.
Зарплаты: Примерно у половины вакансий не указана заработная плата, что может усложнить обучение модели. Средняя «вилка» составляет примерно от 71 000 до 110 000 рублей.
Тип занятости: Самое популярное сочетание - «Полный день - Полная занятость». Это говорит о том, что работодатели по-прежнему предпочитают стандартный формат работы для большинства позиций.
Опыт: Наибольший спрос наблюдается на специалистов с опытом от 1 до 3 лет (Middle), а меньше всего вакансий для соискателей с опытом более 6 лет, что характерно для высокой востребованности специалистов среднего звена.

# Юнит 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [23]:
query_5_1 = '''
    SELECT 
        e.name, 
        COUNT(v.id) AS cnt
    FROM 
        employers e
    JOIN 
        vacancies v ON e.id = v.employer_id
    GROUP BY 
        e.name
    ORDER BY 
        cnt DESC
    LIMIT 5
'''

In [24]:
df = pd.read_sql_query(query_5_1, connection)
df

,name,cnt
0,Яндекс,1933
1,Ростелеком,491
2,Тинькофф,444
3,СБЕР,428
4,Газпром нефть,331


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [25]:
query_5_2 = '''
    SELECT 
        a.name AS area_name,
        COUNT(DISTINCT e.id) AS employers_cnt,
        COUNT(v.id) AS vacancies_cnt
    FROM 
        areas a
    LEFT JOIN 
        employers e ON a.id = e.area
    LEFT JOIN 
        vacancies v ON a.id = v.area_id
    GROUP BY 
        a.id
    HAVING 
        COUNT(v.id) = 0
    ORDER BY 
        employers_cnt DESC
    LIMIT 1
'''

In [26]:
df = pd.read_sql_query(query_5_2, connection)
df

,area_name,employers_cnt,vacancies_cnt
0,Россия,410,0


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [27]:
query_5_3 = '''
    SELECT 
        e.name,
        COUNT(DISTINCT v.area_id) AS areas_cnt
    FROM 
        employers e
    JOIN 
        vacancies v ON e.id = v.employer_id
    GROUP BY 
        e.id
    ORDER BY 
        areas_cnt DESC
'''

In [28]:
df = pd.read_sql_query(query_5_3, connection)
df

,name,areas_cnt
0,Яндекс,181
1,Ростелеком,152
2,Спецремонт,116
3,Поляков Денис Иванович,88
4,ООО ЕФИН,71
...,...,...
14901,НПП Авиатрон,1
14902,Центр дистанционных торгов,1
14903,Городские Телекоммуникационные Системы,1
14904,"Введенский, Отель",1


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [29]:
query_5_4 = '''
    SELECT 
        COUNT(*)
    FROM 
        employers e
    LEFT JOIN 
        employers_industries ei ON e.id = ei.employer_id
    WHERE 
        ei.industry_id IS NULL
'''

In [30]:
df = pd.read_sql_query(query_5_4, connection)
df

,count
0,8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [31]:
query_5_5 = '''
    SELECT 
        e.name
    FROM 
        employers e
    JOIN 
        employers_industries ei ON e.id = ei.employer_id
    GROUP BY 
        e.id
    HAVING 
        COUNT(ei.industry_id) = 4
    ORDER BY 
        e.name ASC
    OFFSET 2
    LIMIT 1
'''

In [32]:
df = pd.read_sql_query(query_5_5, connection)
df

,name
0,2ГИС


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [33]:
query_5_6 = '''
    SELECT 
        COUNT(*)
    FROM 
        employers_industries ei
    JOIN 
        industries i ON ei.industry_id = i.id
    WHERE 
        i.name = 'Разработка программного обеспечения'
'''

In [34]:
df = pd.read_sql_query(query_5_6, connection)
df

,count
0,3553


7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [35]:
url = 'https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
html_data = io.StringIO(response.text)
tables = pd.read_html(html_data)
cities_table = tables[0]
cities = tuple(cities_table['Город'].tolist())

In [36]:
query_5_7 = f'''
    WITH yandex_vacancies AS (
        SELECT 
            a.name AS city,
            COUNT(*) AS cnt
        FROM 
            vacancies v
        JOIN 
            employers e ON v.employer_id = e.id
        JOIN 
            areas a ON v.area_id = a.id
        WHERE 
            e.name = 'Яндекс' AND a.name IN {cities}
        GROUP BY 
            a.name
    )
    
    SELECT city, cnt FROM yandex_vacancies
    UNION ALL
    SELECT 'Total', SUM(cnt) FROM yandex_vacancies
    ORDER BY cnt ASC
'''

In [37]:
df = pd.read_sql_query(query_5_7, connection)
df

,city,cnt
0,Омск,21.0
1,Челябинск,22.0
2,Красноярск,23.0
3,Волгоград,24.0
4,Пермь,25.0
5,Казань,25.0
6,Ростов-на-Дону,25.0
7,Самара,26.0
8,Уфа,26.0
9,Краснодар,30.0


***

Лидеры рынка: Крупнейшим работодателем является Яндекс, значительно опережающий конкурентов по количеству вакансий. В топ-5 также входят компании нефтегазового сектора (Газпром нефть) и банковской сферы (Тинькофф), что указывает на высокую активность IT-найма в этих отраслях.
Географическая экспансия: Компании-гиганты (тот же Яндекс) размещают вакансии в сотнях регионов, включая города-миллионники. Это подтверждает как развитую региональную сеть, так и готовность к удаленному формату работы.
Специфика регионов: Существуют регионы (например, Россия в целом), где зарегистрировано много работодателей, но нет конкретных вакансий. Это может быть связано с тем, что компании указывают головной офис в одном месте, а вакансии распределяют по конкретным городам.
Индустрии: Около трети работодателей не указывают сферу деятельности, однако среди тех, кто указывает, лидирует «Разработка программного обеспечения». Компании часто совмещают несколько направлений (в среднем до 4-х), что характерно для крупных экосистем.

# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [38]:
query_6_1 = '''
    SELECT 
        COUNT(*)
    FROM 
        vacancies
    WHERE 
        LOWER(name) LIKE '%data%' 
        OR 
        LOWER(name) LIKE '%данн%'
'''

In [39]:
df = pd.read_sql_query(query_6_1, connection)
df

,count
0,1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [40]:
query_6_2 = '''
    SELECT 
        COUNT(*)
    FROM 
        vacancies
    WHERE 
        (
            name ILIKE '%data scientist%' OR 
            name ILIKE '%data science%' OR 
            name ILIKE '%исследователь данных%' OR 
            (name LIKE '%ML%' AND name NOT ILIKE '%HTML%') OR 
            name ILIKE '%machine learning%' OR 
            name ILIKE '%машинн%обучен%'
        )
        AND 
        (
            name ILIKE '%junior%' OR 
            experience = 'Нет опыта' OR 
            employment = 'Стажировка'
        )
'''

In [41]:
df = pd.read_sql_query(query_6_2, connection)
df

,count
0,51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [42]:
query_6_3 = '''
    SELECT 
        COUNT(*)
    FROM 
        vacancies
    WHERE 
        (
            name ILIKE '%data scientist%' OR 
            name ILIKE '%data science%' OR 
            name ILIKE '%исследователь данных%' OR 
            (name ILIKE '%ML%' AND name NOT ILIKE '%HTML%') OR 
            name ILIKE '%machine learning%' OR 
            name ILIKE '%машинн%обучен%'
        )
        AND 
        (
            key_skills ILIKE '%SQL%' OR 
            key_skills ILIKE '%Postgre%'
        )
'''

In [43]:
df = pd.read_sql_query(query_6_3, connection)
df

,count
0,229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [44]:
query_6_4 = '''
    SELECT 
        COUNT(*)
    FROM 
        vacancies
    WHERE 
        (
            name ILIKE '%data scientist%' OR 
            name ILIKE '%data science%' OR 
            name ILIKE '%исследователь данных%' OR 
            (name ILIKE '%ML%' AND name NOT ILIKE '%HTML%') OR
            name ILIKE '%machine learning%' OR 
            name ILIKE '%машинн%обучен%'
        )
        AND 
        (
            key_skills ILIKE '%Python%'
        )
'''

In [45]:
df = pd.read_sql_query(query_6_4, connection)
df

,count
0,357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [46]:
query_6_5 = '''
    SELECT 
        ROUND(AVG(LENGTH(key_skills) - LENGTH(REPLACE(key_skills, '\t', '')) + 1), 2)
    FROM 
        vacancies
    WHERE 
        (
            name ILIKE '%data scientist%' OR 
            name ILIKE '%data science%' OR 
            name ILIKE '%исследователь данных%' OR 
            (name LIKE '%ML%' AND name NOT ILIKE '%HTML%') OR 
            name ILIKE '%machine learning%' OR 
            name ILIKE '%машинн%обучен%'
        )
        AND 
        key_skills IS NOT NULL
'''

In [47]:
df = pd.read_sql_query(query_6_5, connection)
df

,round
0,6.41


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [48]:
query_6_6 = '''
    SELECT 
        experience,
        ROUND(AVG((COALESCE(salary_from, salary_to) + COALESCE(salary_to, salary_from)) / 2)) AS avg_salary
    FROM 
        vacancies
    WHERE 
        (
            name ILIKE '%data scientist%' OR 
            name ILIKE '%data science%' OR 
            name ILIKE '%исследователь данных%' OR 
            (name LIKE '%ML%' AND name NOT ILIKE '%HTML%') OR 
            name ILIKE '%machine learning%' OR 
            name ILIKE '%машинн%обучен%'
        )
        AND 
        (salary_from IS NOT NULL OR salary_to IS NOT NULL)
    GROUP BY 
        experience
'''

In [49]:
df = pd.read_sql_query(query_6_6, connection)
df

,experience,avg_salary
0,Нет опыта,74643.0
1,От 1 года до 3 лет,139675.0
2,От 3 до 6 лет,243115.0


***

Востребованность: Вакансии, связанные с данными, составляют заметную часть IT-рынка, при этом узкая специализация Data Scientist представлена примерно в 1700+ вакансиях (в зависимости от строгости фильтра).
Инструментарий: Python является абсолютным лидером среди навыков для DS - он требуется практически в каждой второй вакансии этой категории. SQL и Postgres также критически важны и встречаются примерно в каждой восьмой вакансии для специалистов по данным.
Портрет кандидата: Работодатели ищут специалистов с широким набором компетенций - в среднем в вакансии DS указывается около 6.41 ключевых навыков, что выше, чем в среднем по другим IT-направлениям.
Зарплатные ожидания: В сфере Data Science наблюдается одна из самых высоких динамик роста зарплат в зависимости от опыта. Специалисты с опытом от 3 до 6 лет могут рассчитывать на зарплату в среднем около 243 тысяч рублей, что делает это направление одним из самых высокооплачиваемых.
Junior-позиции: Доля вакансий для начинающих специалистов (без опыта или стажировка) невелика, что говорит о высоком пороге входа и предпочтении компаниями кандидатов с уже имеющимся практическим опытом.

# Общий вывод по проекту

В ходе анализа данных HeadHunter были сделаны следующие ключевые наблюдения:
География и рынок: Рынок IT-вакансий сильно централизован (Москва и Санкт-Петербург), однако крупные технологические компании (Яндекс, Сбер, Тинькофф) активно расширяют присутствие в регионах-миллионниках.
Профиль Data Scientist: Это высокооплачиваемая роль с порогом входа от 1-3 лет опыта. Ключевой стек - Python и SQL. Высокое среднее количество навыков (более 6) подтверждает междисциплинарный характер профессии.
Качество данных: Около половины вакансий не имеют указанной зарплаты, что является вызовом для построения модели машинного обучения. Для обучения будущей модели рекомендуется использовать медианные значения зарплат по регионам и опыту.